# This is a demonstration of Synthesizer
## What is Grid object?

In [3]:
from synthesizer import GRID_DIR
print(GRID_DIR)

/Users/jiaheak/Library/Application Support/Synthesizer/grids


In [2]:
import matplotlib.pyplot as plt
import numpy as np
from synthesizer import Grid
# Return to the unmodified grid
grid = Grid("test_grid")

log10age = 6.0  # log10(age/yr)
Z = 0.01  # metallicity
grid_point = grid.get_grid_point(log10ages=log10age, metallicities=Z)

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = '/Users/jiaheak/Library/Application Support/Synthesizer/grids/test_grid.hdf5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [ ]:
for spectra_type in grid.available_spectra:
    # Get `Sed` object
    sed = grid.get_sed_at_grid_point(grid_point, spectra_type=spectra_type)

    # Mask zero valued elements
    mask = sed.lnu > 0
    plt.plot(
        np.log10(sed.lam[mask]),
        np.log10(sed.lnu[mask]),
        lw=1,
        alpha=0.8,
        label=spectra_type,
    )

plt.legend(fontsize=8, labelspacing=0.0)
plt.xlim(2.3, 8)
plt.ylim(19, 25)
plt.xlabel(r"$\rm log_{10}(\lambda/\AA)$")
plt.ylabel(r"$\rm log_{10}(L_{\nu}/erg\ s^{-1}\ Hz^{-1} M_{\odot}^{-1})$")

# Galaxies & Components
## Particle Stars

In [ ]:
from synthesizer.particle import Stars, Gas, BlackHoles, Galaxy

N = 100
ages = np.random.rand(N) * 100 * Myr
metallicities = np.random.rand(N) / 100
initial_masses = np.ones(N) * 1e6 * Msun

stars = Stars(
    initial_masses=initial_masses,
    ages=ages,
    metallicities=metallicities,
)

stars.plot_sfh(grid.log10ages)
stars.plot_metal_dist(grid.metallicities)

In [ ]:
gas = Gas(
    masses=np.ones(1000) * 10**6 * Msun,
    metallicities=np.random.rand(1000) * 0.02,
    dust_to_metal_ratio=0.25,
    coordinates=np.random.rand(1000, 3) * 1 * Mpc,
    centre=np.mean(np.random.rand(1000, 3) * 1, axis=0) * Mpc,
    hii_mass=np.random.rand(1000) * 1e4 * Msun,
    hii_metallicity=np.random.rand(1000) * 0.02,
)
print(gas)

In [ ]:
# Make fake properties
n = 4
masses = 10 ** np.random.uniform(low=7, high=9, size=n) * Msun
coordinates = np.random.normal(0, 1.5, (n, 3)) * Mpc
accretion_rates = 10 ** np.random.uniform(low=-2, high=1, size=n) * Msun / yr
metallicities = np.full(n, 0.01)

# And get the black holes object
bh = BlackHoles(
    masses=masses,
    coordinates=coordinates,
    accretion_rates=accretion_rates,
    metallicities=metallicities,
)

## parametric Star Formation History

In [ ]:
from unyt import yr
from synthesizer.parametric import SFH

sfh = SFH.DoublePowerLaw(
    peak_age=5e8 * yr, alpha=10, beta=-10, max_age=1e9 * yr
)

sfh.plot_sfh(t_range=(0, 1e9))

# Emissions


In [ ]:
from unyt import Angstrom, Hz, erg, eV, s, um
from synthesizer import Sed

# Define some wavelengths and luminosities densities
lams = np.logspace(3, 5, 100) * Angstrom
lnus = np.logspace(26, 30, 100) * erg / (s * Hz)

# Create a Sed object
sed = Sed(lams, lnus)

In [ ]:
print(sed.wavelength)
print(sed.luminosity_nu)

In [ ]:
plt.plot(np.log10(sed.lam), np.log10(sed.lnu))
plt.show()

In [ ]:
fig, ax = sed.plot_spectra_as_rainbow()

In [ ]:
from synthesizer.emission_models.attenuation import PowerLaw

sed4_att = sed4.apply_attenuation(tau_v=0.7, dust_curve=PowerLaw(-1.0))

# Integrate the multidimensional spectra
int_sed4 = sed4.sum()
int_sed4_att = sed4_att.sum()

fig, ax = int_sed4.plot_spectra(label="Original")
fig, ax = int_sed4_att.plot_spectra(label="Attenuated", fig=fig, ax=ax)